# Geosteering AI — Sprint O0/O1: Validação JAX GPU pós-Quick Wins

**Template:** `validate_sprint_o0_o1_gpu.ipynb` | **Sprint:** O0 (anti-regressão) + O1 (Quick Wins)
**Branch alvo:** `feat/sprint-o1-quick-wins` (deve estar **pushada** no remoto)

Adaptado de `validate_jax_gpu_v240.ipynb` (A1.6) — preserva o padrão estabelecido
e descarta as seções de benchmark Numba-vs-JAX específicas da A1.6.

## ⚠️ Pré-requisito BLOQUEANTE

A branch `feat/sprint-o1-quick-wins` precisa estar pushada no GitHub **antes** de
executar este notebook. Caso contrário, a célula de clone falhará — propositalmente,
para evitar o fallback silencioso para `main` que mascarou o erro na execução anterior.

```bash
# No host de desenvolvimento:
git push -u origin feat/sprint-o1-quick-wins
```

## O que este notebook valida

| § | Seção | Critério |
|:-:|:------|:---------|
| 1-2 | Configurações + Setup | GPU CUDA, float64, branch correta clonada |
| 3 | Quick Wins #1–#9 ativos | 8 verificações: imports + env vars + config flags |
| 4 | Suite GPU `pytest -m gpu` | ≈107 testes, 0 FAILED (paridade Fortran <1e-12 + T1.1) |
| 5 | Sanity batched API | Shape (n,1,1,n_pos,1,9), dtype complex128, sem NaN/Inf |
| 6 | Gate T1.6 | Throughput hot ≥ 90% baseline A100 em A, B, E |
| 7 | Atualização baseline (opcional) | `UPDATE_BASELINE=True` regrava `.claude/perf_baseline.json` |
| 8 | Export Drive | JSON consolidado com pytest + Quick Wins + T1.6 + throughputs |
| 9 | Decisão Go/No-Go | Aprovação para merge `feat/sprint-o1-quick-wins → main` |
| A | (opt-in) Profiling roofline | Trace `.json.gz` para perfetto.dev (Sprint O2+ guidance) |

## Quick Wins Sprint O1 verificados

| # | Otimização | API/Símbolo verificado |
|:-:|:-----------|:----------------------|
| 1 | `_layer_at_depth_vec` (NumPy vetorizado) | `from ...multi_forward import _layer_at_depth_vec` |
| 2 | `lax.scan(unroll=K)` em propagação/dipolos | `SimulationConfig().unroll_layer_loop == True` |
| 3 | XLA persistent compilation cache | `JAX_COMPILATION_CACHE_DIR` setado |
| 4 | XLA flag `latency_hiding_scheduler` | `XLA_FLAGS` contém a flag |
| 5 | `jax.config.jax_compilation_cache_dir` (API moderna) | Via Quick Win #3 |
| 6 | Hankel LRU cache process-wide | 2ª chamada ≥10× mais rápida que 1ª |
| 7 | Sync GPU→CPU deferido | (caminho interno, validado via T1.6) |
| 8 | `jax.tree.map` consolida HtoD transfers | (caminho interno, validado via T1.6) |
| 9 | `_sanitize_profile_batch` vetorizado | `from ...multi_forward import _sanitize_profile_batch` |


In [ ]:
# === L5: Configurações JAX OBRIGATÓRIAS antes de qualquer import ===
# Equivalente ao 'NUMBA_NUM_THREADS no PAI' (L5 do mapeamento Numba→JAX):
# jax_enable_x64 e jax_compilation_cache_dir devem ser definidos ANTES
# de 'import jax' — após o import, jax_enable_x64 fica imutável.
import os

os.environ["JAX_ENABLE_X64"] = "1"                              # float64 — paridade Fortran <1e-12 requer float64
os.environ["JAX_PLATFORMS"] = "cuda,cpu"                        # CUDA preferido, CPU fallback (compat JAX 0.4.30+)
os.environ["JAX_COMPILATION_CACHE_DIR"] = "/content/jax_cache"  # L8 — cache XLA persistente em SSD Colab

# Quick Win #4 — XLA flags A100/T4 (latency hiding scheduler)
# IMPORTANTE: xla_gpu_enable_triton_softmax_fusion foi REMOVIDA em JAX ≥0.5/XLA ≥0.7.
# Setar flag desconhecida causa FATAL em parse_flags_from_env.cc e mata o kernel Colab.
_existing_xla = os.environ.get("XLA_FLAGS", "")
_o1_flags = ["--xla_gpu_enable_latency_hiding_scheduler=true"]
os.environ["XLA_FLAGS"] = " ".join(
    [_existing_xla] + [f for f in _o1_flags if f not in _existing_xla]
).strip()

# Quick Win #4 — VRAM allocator (BFC dinâmico — evita pré-alocação eager de 75%)
# NOTA: XLA_PYTHON_CLIENT_MEM_FRACTION só tem efeito com PREALLOCATE=true.
# Com PREALLOCATE=false (BFC dinâmico), a fração é IGNORADA — mantida abaixo
# apenas como cap de segurança caso alguém troque para preallocate=true.
os.environ.setdefault("XLA_PYTHON_CLIENT_PREALLOCATE", "false")
os.environ.setdefault("XLA_PYTHON_CLIENT_MEM_FRACTION", "0.85")

print("JAX env vars configurados ANTES de qualquer import jax:")
for k in ("JAX_ENABLE_X64", "JAX_PLATFORMS", "JAX_COMPILATION_CACHE_DIR",
          "XLA_FLAGS", "XLA_PYTHON_CLIENT_PREALLOCATE", "XLA_PYTHON_CLIENT_MEM_FRACTION"):
    print(f"  {k:32s} = {os.environ.get(k, '(não setado)')!r}")


## § 1 — Configurações da Sprint

In [ ]:
# Branch a clonar — DEVE ESTAR PUSHADA NO REMOTO
# (sem fallback para main: queremos fail-fast se branch não existir)
GIT_BRANCH       = "feat/sprint-o1-quick-wins"
GIT_REPO_URL     = "https://github.com/daniel-guitarplayer-8/geosteering-ai.git"
REPO_DIR         = "/content/geosteering-ai"
OUTPUT_DRIVE_DIR = "/content/drive/MyDrive/Geosteering_AI/sprint_o0_o1/"

# Configuração T1.6 — gate de regressão de throughput
# CRÍTICO: N_RUNS_HOT=3 espelha tests/test_simulation_jax_perf_baseline.py:117
# (n_runs_hot: int = 3). Não alterar sem atualizar o teste oficial — mediana
# de 3 vs 4 runs introduz viés estatístico (mediana de 4 = média dos 2 centrais).
N_RUNS_HOT       = 3       # match exato com _measure_throughput_mod_h
SEED             = 42      # reprodutibilidade — match T1.6 oficial

# Update baseline: se True, sobrescreve seção jax_gpu_a100 em .claude/perf_baseline.json
# com as medições pós-O1. Use False para apenas comparar contra baseline existente.
UPDATE_BASELINE  = False   # mude para True somente após confirmar O1 PASS

# Cenários do gate T1.6 (subset oficial — A, B, E)
# Match exato com test_simulation_jax_perf_baseline.py:_SCENARIOS_GATE
T16_GATE_SCENARIOS = {
    "A": {"n_models": 50, "n_pos": 1,   "n_freqs": 1, "n_tr": 1, "n_ang": 1},
    "B": {"n_models": 50, "n_pos": 100, "n_freqs": 1, "n_tr": 1, "n_ang": 1},
    "E": {"n_models": 50, "n_pos": 600, "n_freqs": 1, "n_tr": 1, "n_ang": 1},
}

# Cria diretório cache JAX antes de qualquer import jax
os.makedirs("/content/jax_cache", exist_ok=True)

print(f"GIT_BRANCH       = {GIT_BRANCH!r}")
print(f"N_RUNS_HOT       = {N_RUNS_HOT} (+1 warmup descartado)")
print(f"SEED             = {SEED}")
print(f"UPDATE_BASELINE  = {UPDATE_BASELINE}")
print(f"T1.6 cenários    = {list(T16_GATE_SCENARIOS)}")


## § 2 — Setup do Ambiente

Monta Drive, clona a branch alvo (**fail-fast** se não existir no remoto),
instala dependências e valida que a GPU CUDA está disponível com float64.

**Importante:** instala via `pip install -e ".[all]"` (NÃO `[dev]`) — `[dev]` inclui
`pytest-qt` que aborta a coleta pytest sem Qt bindings (problema do notebook anterior).
`pytest-json-report` é instalado separadamente para parsing estruturado da suite GPU.


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

# Clone — fail-fast se branch não estiver no remoto (sem fallback silencioso)
import subprocess, sys, shutil

if os.path.exists(REPO_DIR):
    shutil.rmtree(REPO_DIR)

print(f"Clonando branch {GIT_BRANCH!r} do remoto ...")
_clone = subprocess.run(
    ["git", "clone", "--depth=1", "--branch", GIT_BRANCH, GIT_REPO_URL, REPO_DIR],
    capture_output=True, text=True,
)
if _clone.returncode != 0:
    raise RuntimeError(
        f"FALHA AO CLONAR branch {GIT_BRANCH!r}.\n"
        f"stderr: {_clone.stderr.strip()}\n\n"
        f"Verifique se a branch foi pushada com:\n"
        f"  git push -u origin {GIT_BRANCH}\n"
        f"Este notebook NÃO faz fallback para main propositalmente — "
        f"queremos validar a branch O0/O1, não main."
    )

# Instalar pacote + extras [all] (sem [dev] → sem pytest-qt → sem erro Qt no Colab)
print("Instalando geosteering_ai[all] + pytest-json-report ...")
_install = subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-e", f"{REPO_DIR}[all]"],
    capture_output=True, text=True,
)
if _install.returncode != 0:
    print(_install.stdout[-2000:])
    print(_install.stderr[-2000:])
    raise RuntimeError("pip install -e .[all] FALHOU")

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q",
     "pytest>=7", "pytest-json-report>=1.5"],
    check=True,
)

# Adicionar repo ao sys.path antes de qualquer import geosteering_ai
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

# Pin commit hash (imutável)
GIT_COMMIT = subprocess.check_output(
    ["git", "rev-parse", "HEAD"], cwd=REPO_DIR
).decode().strip()
print(f"✓ Repositório clonado em commit {GIT_COMMIT[:12]}")

# Imports — apenas APÓS as env vars da Célula 1
import json
import datetime
import re
import statistics
import time
from pathlib import Path
from typing import Optional

import jax
import jax.numpy as jnp
import numpy as np

# Validar GPU — usar jax.default_backend() (API estável JAX 0.4.14+)
_BACKEND = jax.default_backend()
_DEVICES = jax.devices()
_GPU_DEVS = [d for d in _DEVICES if "gpu" in str(d).lower() or "cuda" in str(d).lower()]

IS_GPU = _BACKEND in ("gpu", "cuda") and bool(_GPU_DEVS)
assert IS_GPU, (
    f"GPU NÃO detectada. backend={_BACKEND!r}, devices={_DEVICES}.\n"
    "Verifique: Runtime → Change runtime type → GPU (T4 ou A100)."
)
print(f"Backend default: {_BACKEND!r} ✓")
print(f"Dispositivos GPU: {_GPU_DEVS}")

assert jax.config.jax_enable_x64, (
    "ERRO CRÍTICO: float64 não habilitado! "
    "JAX_ENABLE_X64 deve ser setado ANTES de 'import jax' (Célula 1)."
)
print(f"float64 habilitado: {jax.config.jax_enable_x64} ✓")

# Hardware info — detecta T4 vs A100
_gpu_name_raw = subprocess.check_output(
    ["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv,noheader"]
).decode().strip()
print(f"\n[GPU] {_gpu_name_raw}")
_gpu_name_lower = _gpu_name_raw.lower()
RUNTIME_LABEL = "a100" if "a100" in _gpu_name_lower else ("t4" if "t4" in _gpu_name_lower else "gpu_unknown")
print(f"RUNTIME_LABEL    = {RUNTIME_LABEL!r}")
print(f"JAX version      = {jax.__version__}")


## § 3 — Verificação dos Quick Wins Sprint O1

Confirma que os 9 Quick Wins estão ativos e configurados corretamente
**antes** de rodar a suite de testes. Os checks são informativos (não bloqueiam
a suite pytest) — qualquer FAIL aqui é registrado no JSON de export.


In [ ]:
# Quick Wins #1–#9 — verificação ativa
qw_checks = []

# ── #3 XLA persistent cache dir ───────────────────────────────────────────────
_cache_dir = os.environ.get("JAX_COMPILATION_CACHE_DIR", "")
qw_checks.append(("#3 XLA cache dir setado", bool(_cache_dir), _cache_dir or "(não setado)"))

# ── #4 XLA flag latency_hiding_scheduler ──────────────────────────────────────
_xla = os.environ.get("XLA_FLAGS", "")
qw_checks.append(
    ("#4 XLA flag latency_hiding_scheduler",
     "xla_gpu_enable_latency_hiding_scheduler=true" in _xla,
     _xla[:120] or "(não setado)")
)

# ── #4 VRAM allocator config ──────────────────────────────────────────────────
qw_checks.append(
    ("#4 XLA_PYTHON_CLIENT_PREALLOCATE=false",
     os.environ.get("XLA_PYTHON_CLIENT_PREALLOCATE") == "false",
     os.environ.get("XLA_PYTHON_CLIENT_PREALLOCATE", "(não setado)"))
)
qw_checks.append(
    ("#4 XLA_PYTHON_CLIENT_MEM_FRACTION=0.85",
     os.environ.get("XLA_PYTHON_CLIENT_MEM_FRACTION") == "0.85",
     os.environ.get("XLA_PYTHON_CLIENT_MEM_FRACTION", "(não setado)"))
)

# ── #5 jax.config.jax_compilation_cache_dir (API estável) ─────────────────────
# Usa getattr (não jax.config.read — esta é privada/instável em JAX 0.7.x).
# Em algumas versões o env var não é propagado para jax.config até a 1ª compile;
# tratamos ausência como WARN, não FAIL.
_jax_cache_cfg = getattr(jax.config, "jax_compilation_cache_dir", None) or ""
_qw5_ok = bool(_cache_dir)  # critério: env var setado é suficiente (XLA lê dele)
qw_checks.append(
    ("#5 jax.config.jax_compilation_cache_dir (env var propagado)",
     _qw5_ok,
     _jax_cache_cfg or f"(jax.config não expõe; env={_cache_dir!r})")
)

# ── #2 SimulationConfig.unroll_layer_loop default ─────────────────────────────
try:
    from geosteering_ai.simulation.config import SimulationConfig
    _cfg = SimulationConfig()
    _unroll = getattr(_cfg, "unroll_layer_loop", None)
    qw_checks.append(
        ("#2 SimulationConfig.unroll_layer_loop default=True",
         _unroll is True,
         f"unroll_layer_loop={_unroll!r}")
    )
except Exception as _e:
    qw_checks.append(("#2 SimulationConfig.unroll_layer_loop", False, f"import erro: {_e}"))

# ── #6 Hankel LRU cache process-wide ──────────────────────────────────────────
# Para medir cache real (não só dispatch async JAX), sincroniza cada chamada
# com .block_until_ready() — sem isso, time.perf_counter() mede apenas o
# enqueue do device put, falsificando o speedup.
try:
    from geosteering_ai.simulation._jax.kernel import get_hankel_filter_jax
    _t0 = time.perf_counter()
    _kr1, _w0_1, _w1_1 = get_hankel_filter_jax("werthmuller_201pt")
    _kr1.block_until_ready()  # força sync HtoD da 1ª chamada
    _t1 = time.perf_counter()
    _kr2, _w0_2, _w1_2 = get_hankel_filter_jax("werthmuller_201pt")
    _kr2.block_until_ready()  # força sync (cache hit não faz HtoD)
    _t2 = time.perf_counter()
    _first_ms  = (_t1 - _t0) * 1000
    _second_ms = (_t2 - _t1) * 1000
    _speedup   = _first_ms / max(_second_ms, 1e-6)
    qw_checks.append(
        (f"#6 Hankel LRU cache speedup (2ª chamada)",
         _speedup > 10,
         f"1ª={_first_ms:.1f}ms → 2ª={_second_ms:.3f}ms (speedup {_speedup:.0f}×)")
    )
except Exception as _e:
    qw_checks.append(("#6 Hankel LRU cache", False, f"import/call erro: {_e}"))

# ── #1 _layer_at_depth_vec (NumPy vetorizado) ─────────────────────────────────
# Smoke call replicando o docstring example (multi_forward.py:344):
#   prof_mid shape (n,) — n=3 camadas → boundaries [0.0, 1.0, 2.0]
#   z=[-1.0, 0.5, 1.5, 100.0] → esperado [0, 0, 1, 2] (RX-inclusive >=)
try:
    from geosteering_ai.simulation._jax.multi_forward import _layer_at_depth_vec
    _z = np.array([-1.0, 0.5, 1.5, 100.0])
    _prof = np.array([0.0, 1.0, 2.0])  # shape (n,) = (3,)
    _idx = _layer_at_depth_vec(3, _z, _prof)
    _expected = [0, 0, 1, 2]
    _ok = _idx.shape == (4,) and _idx.tolist() == _expected
    qw_checks.append(
        ("#1 _layer_at_depth_vec funcional + RX-inclusive",
         _ok,
         f"idx={_idx.tolist()} (esperado {_expected})")
    )
except Exception as _e:
    qw_checks.append(("#1 _layer_at_depth_vec", False, f"import/call erro: {_e}"))

# ── #9 _sanitize_profile_batch importável ─────────────────────────────────────
try:
    from geosteering_ai.simulation._jax.multi_forward import _sanitize_profile_batch
    qw_checks.append(("#9 _sanitize_profile_batch importável", True, "OK"))
except Exception as _e:
    qw_checks.append(("#9 _sanitize_profile_batch", False, f"import erro: {_e}"))

# ── Relatório ─────────────────────────────────────────────────────────────────
print("=" * 75)
print("Sprint O1 — Verificação de Quick Wins")
print("=" * 75)
_qw_n_pass = 0
for label, ok, detail in qw_checks:
    status = "✓ PASS" if ok else "✗ FAIL"
    if ok:
        _qw_n_pass += 1
    print(f"  {status} {label}")
    if not ok or "speedup" in label.lower():
        print(f"         {detail}")

print("=" * 75)
QW_ALL_PASS = _qw_n_pass == len(qw_checks)
print(f"  {_qw_n_pass}/{len(qw_checks)} Quick Wins confirmados {'✓' if QW_ALL_PASS else '⚠'}")


## § 4 — Suite GPU completa: `pytest tests/test_simulation_jax_*.py -m gpu`

Roda **todos** os testes `@pytest.mark.gpu` em arquivos `test_simulation_jax_*.py`
(≈107 testes — paridade Fortran <1e-12 + T1.1 + Sprint 10/12/13/dipoles/etc).
Inclui automaticamente os arquivos novos do Sprint O0 (`test_simulation_jax_fortran_parity.py`,
`test_simulation_jax_perf_baseline.py`).

**Critério de gate:** 0 FAILED + ≥1 PASSED (evita o bug do notebook anterior que
aceitava `0 passed AND 0 failed` como sucesso quando a coleta abortava silenciosamente).

**Importante:** este passo usa `!python -m pytest` (magic Jupyter) e `%cd` em vez de
`subprocess.run + cwd=` — padrão estabelecido pelo notebook A1.6, comprovadamente
funcional no Colab T4.


In [ ]:
# Mudar para o diretório do repo (necessário para pytest resolver tests/conftest.py)
# AVISO: %cd altera o cwd GLOBAL da sessão Jupyter — persiste em todas as
# células subsequentes. Sempre use Path(REPO_DIR) (absoluto) em código,
# nunca caminhos relativos, para que o notebook seja robusto a re-execuções
# de células fora de ordem.
%cd /content/geosteering-ai

# Executa todos os testes JAX GPU-only e gera JSON estruturado
!python -m pytest tests/test_simulation_jax_*.py \
    -m gpu \
    -v \
    --tb=short \
    --json-report \
    --json-report-file=/tmp/jax_gpu_pytest.json \
    -p no:cacheprovider \
    2>&1 | tail -60

# Carregar relatório pytest UMA VEZ — reutilizado em § 8 e § 9
with open("/tmp/jax_gpu_pytest.json") as _f:
    _pytest_report = json.load(_f)
_summary = _pytest_report["summary"]
print(f"\nResumo pytest: {_summary}")

# Gate robusto: exige passed > 0 E failed == 0 (corrige bug do notebook anterior)
GPU_SUITE_PASS = (
    _summary.get("failed", 1) == 0
    and _summary.get("errors", 0) == 0
    and _summary.get("passed", 0) > 0
)
if not GPU_SUITE_PASS:
    print(
        f"\n✗ SUITE GPU FALHOU OU NÃO COLETOU TESTES.\n"
        f"   passed={_summary.get('passed', 0)} | "
        f"failed={_summary.get('failed', 0)} | "
        f"errors={_summary.get('errors', 0)} | "
        f"skipped={_summary.get('skipped', 0)}"
    )
else:
    print(f"\n✓ Suite GPU: {_summary['passed']} PASSED / 0 FAILED ({_summary.get('skipped', 0)} skipped)")


## § 5 — Sanity check da API batched

Confirma forma, dtype, ausência de NaN/Inf antes de medir throughput.
Não é gate formal — apenas defesa contra regressões grosseiras.


In [ ]:
from geosteering_ai.simulation import simulate_multi_jax_batched

_n_check = 3
_rho_h_check = np.tile(np.array([1.0, 10.0, 100.0, 10.0, 1.0]), (_n_check, 1))
_rho_v_check = _rho_h_check * 2.0  # TIV anisotrópico
_esp_check   = np.tile(np.array([5.0, 10.0, 5.0]), (_n_check, 1))
_pos_check   = np.linspace(-5.0, 5.0, 50).astype(np.float64)

_res = simulate_multi_jax_batched(
    _rho_h_check, _rho_v_check, _esp_check, _pos_check,
    frequencies_hz=[20000.0], tr_spacings_m=[1.0], dip_degs=[0.0],
)
# H_tensor já é np.ndarray (simulate_multi_jax_batched chama np.asarray
# internamente em multi_forward.py:1189). Leitura de .shape é no-op de sync.
_ = _res.H_tensor.shape

assert _res.H_tensor.dtype == np.complex128, f"dtype errado: {_res.H_tensor.dtype}"
assert _res.H_tensor.shape == (_n_check, 1, 1, 50, 1, 9), f"shape errado: {_res.H_tensor.shape}"
assert not np.any(np.isnan(_res.H_tensor.real)), "NaN em H_tensor.real!"
assert not np.any(np.isnan(_res.H_tensor.imag)), "NaN em H_tensor.imag!"
assert not np.any(np.isinf(_res.H_tensor)),      "Inf em H_tensor!"

print(f"Shape:  {_res.H_tensor.shape}  ✓")
print(f"dtype:  {_res.H_tensor.dtype}  ✓")
print(f"NaN/Inf: ausentes  ✓")
print(f"|Hzz| médio modelo 0: {np.abs(_res.H_tensor[0,0,0,:,0,8]).mean():.6e}")


## § 6 — Gate T1.6: throughput pós-O1 vs baseline A100

Reproduz a metodologia do teste `test_simulation_jax_perf_baseline.py` (Sprint O0):

1. Carrega seção `jax_gpu_a100` de `.claude/perf_baseline.json` para obter thresholds
2. Para cada cenário gate (A, B, E): mede mediana de `N_RUNS_HOT` runs hot
3. Falha se `medido < threshold_90pct`

A medição usa **exatamente** as mesmas dimensões/modelos do teste oficial
(rng seed 42, 3 camadas isotropicas, `_measure_throughput_mod_h` em
`tests/test_simulation_jax_perf_baseline.py:111`).


In [ ]:
# Carrega baseline jax_gpu_a100 do repositório clonado.
# Tratamento gracioso (não bare assert): degrada para T16_GATE_PASS=False
# em vez de crashear a célula — espelha pytest.skip() do teste oficial.
BASELINE_PATH = Path(REPO_DIR) / ".claude" / "perf_baseline.json"
_baseline_full = None
_baseline_a100 = None
_baseline_load_error = None

if not BASELINE_PATH.exists():
    _baseline_load_error = f"Baseline não encontrada em {BASELINE_PATH}"
else:
    try:
        with BASELINE_PATH.open(encoding="utf-8") as _f:
            _baseline_full = json.load(_f)
        _baseline_a100 = _baseline_full.get("jax_gpu_a100")
        if _baseline_a100 is None:
            _baseline_load_error = "Seção 'jax_gpu_a100' ausente em .claude/perf_baseline.json"
    except (json.JSONDecodeError, OSError) as _exc:
        _baseline_load_error = f"Erro lendo baseline: {_exc}"


def measure_t16_throughput(n_models: int, n_pos: int, n_freqs: int = 1,
                            n_tr: int = 1, n_ang: int = 1,
                            n_runs_hot: int = N_RUNS_HOT,
                            seed: int = SEED) -> dict:
    """Mede throughput hot via simulate_multi_jax_batched.

    Reproduz EXATAMENTE _measure_throughput_mod_h() de
    tests/test_simulation_jax_perf_baseline.py:111-165 — modelo sintético
    oklahoma_3-like (3 camadas isotrópicas), 1 warmup descartado,
    statistics.median sobre runs hot.
    """
    rng = np.random.default_rng(seed)
    rho_h_batch = rng.uniform(1.0, 100.0, size=(n_models, 3)).astype(np.float64)
    rho_v_batch = rho_h_batch.copy()  # isotrópico — match T1.6 oficial
    esp_batch = np.full((n_models, 1), 5.0, dtype=np.float64)
    positions_z = np.linspace(-5.0, 5.0, n_pos).astype(np.float64)
    freqs = np.linspace(1e4, 1e5, n_freqs).astype(np.float64).tolist()
    trs = [1.0 * (i + 1) for i in range(n_tr)]
    dips = [0.0 + 10.0 * i for i in range(n_ang)]

    kwargs = dict(frequencies_hz=freqs, tr_spacings_m=trs, dip_degs=dips)

    # Warmup (descartado). H_tensor já é np.ndarray (sync interno feito).
    _res = simulate_multi_jax_batched(
        rho_h_batch, rho_v_batch, esp_batch, positions_z, **kwargs
    )
    _ = _res.H_tensor.shape  # no-op (já sincronizado por np.asarray no batched API)

    # Runs hot (mediana sobre n_runs_hot=3 — match test_simulation_jax_perf_baseline.py:117)
    elapsed_list = []
    throughputs  = []
    for _ in range(n_runs_hot):
        t0 = time.perf_counter()
        _res = simulate_multi_jax_batched(
            rho_h_batch, rho_v_batch, esp_batch, positions_z, **kwargs
        )
        _ = _res.H_tensor.shape  # no-op (sync já garantido)
        elapsed = time.perf_counter() - t0
        elapsed_list.append(elapsed)
        throughputs.append(n_models / elapsed * 3600.0)

    median_throughput = statistics.median(throughputs)
    return {
        "median_throughput_mod_h": median_throughput,
        "min_throughput_mod_h": min(throughputs),
        "max_throughput_mod_h": max(throughputs),
        "median_elapsed_s": statistics.median(elapsed_list),
        "n_runs_hot": n_runs_hot,
        "throughputs_mod_h": throughputs,
    }


# Roda gate T1.6 para A, B, E
print("=" * 90)
print("GATE T1.6 — throughput pós-O1 vs baseline A100 (threshold = 90% baseline)")
print(f"  Hardware atual: {RUNTIME_LABEL.upper()} | Baseline target: A100")
if RUNTIME_LABEL != "a100":
    print(f"  ⚠ Rodando em {RUNTIME_LABEL.upper()} — baseline jax_gpu_a100 foi medida em A100.")
    print(f"    Gate ainda é executado mas thresholds podem ser inatingíveis em T4.")
print("=" * 90)

t16_results = {}
T16_GATE_PASS = False

if _baseline_a100 is None:
    # SKIP gracioso — espelha pytest.skip() do teste oficial.
    print(f"  ✗ SKIP: {_baseline_load_error}")
    print(f"  Gate T1.6: NÃO AVALIÁVEL (baseline ausente/corrompida)")
else:
    print(f"{'Cenário':>7} | {'n_pos':>5} | {'Baseline (mod/h)':>16} | {'Threshold 90%':>14} | {'Medido (mod/h)':>14} | {'Razão':>6} | {'Status':>8}")
    print("-" * 90)

    t16_all_pass = True
    for scen_name, cfg in T16_GATE_SCENARIOS.items():
        scen_key = f"{scen_name}_hot"
        if scen_key not in _baseline_a100:
            # Cenário ausente é FAIL explícito — não silently skip.
            t16_results[scen_name] = {
                "n_models": cfg["n_models"], "n_pos": cfg["n_pos"],
                "baseline_throughput_mod_h": None, "threshold_90pct_mod_h": None,
                "measured_throughput_mod_h": None, "ratio_vs_baseline": None,
                "passed": False, "skip_reason": f"{scen_key} ausente na baseline",
            }
            t16_all_pass = False
            print(f"{scen_name:>7} | {cfg['n_pos']:>5} | (cenário ausente na baseline) → ✗ FAIL")
            continue

        baseline_throughput = _baseline_a100[scen_key].get("throughput_mod_h")
        threshold = _baseline_a100[scen_key].get("threshold_90pct")

        measurement = measure_t16_throughput(**cfg)
        measured = measurement["median_throughput_mod_h"]
        ratio = measured / baseline_throughput if baseline_throughput else float("nan")
        passed = (threshold is not None) and (measured >= threshold)

        t16_results[scen_name] = {
            "n_models": cfg["n_models"],
            "n_pos": cfg["n_pos"],
            "baseline_throughput_mod_h": baseline_throughput,
            "threshold_90pct_mod_h": threshold,
            "measured_throughput_mod_h": measured,
            "measured_min_mod_h": measurement["min_throughput_mod_h"],
            "measured_max_mod_h": measurement["max_throughput_mod_h"],
            "measured_median_elapsed_s": measurement["median_elapsed_s"],
            "ratio_vs_baseline": ratio,
            "passed": passed,
        }
        t16_all_pass = t16_all_pass and passed

        status_str = "✓ PASS" if passed else "✗ FAIL"
        print(f"{scen_name:>7} | {cfg['n_pos']:>5} | "
              f"{baseline_throughput:>16,.0f} | {threshold:>14,.0f} | "
              f"{measured:>14,.0f} | {ratio:>5.2f}× | {status_str:>8}")

    print("=" * 90)
    T16_GATE_PASS = t16_all_pass and len(t16_results) == len(T16_GATE_SCENARIOS)
    print(f"  Gate T1.6 global: {'✓ APROVADO' if T16_GATE_PASS else '✗ REPROVADO'} "
          f"({sum(r['passed'] for r in t16_results.values())}/{len(T16_GATE_SCENARIOS)} cenários PASS)")


## § 7 — Atualização opcional da baseline (`UPDATE_BASELINE`)

Se `UPDATE_BASELINE=True` (Célula 3) e o gate T1.6 passou, regrava
`.claude/perf_baseline.json` (seção `jax_gpu_a100`) com as medidas atuais.
Salva no Drive uma cópia da baseline anterior para auditoria.

**Quando habilitar:** após confirmar que as otimizações O1 trouxeram ganho real
(medido > baseline + 5%) e que paridade Fortran <1e-12 não regrediu.

**Quando NÃO habilitar:**
- Se gate T1.6 FALHOU (não queremos baixar o bar)
- Se rodando em T4 (baseline oficial é A100)
- Se este for um run exploratório, não o run oficial de release


In [ ]:
if not UPDATE_BASELINE:
    print("UPDATE_BASELINE=False → baseline NÃO atualizada.")
    print("Para atualizar: setar UPDATE_BASELINE=True na Célula 3 e re-rodar.")
    BASELINE_UPDATED = False
elif not T16_GATE_PASS:
    print("⚠ UPDATE_BASELINE=True mas gate T1.6 NÃO passou — baseline NÃO atualizada.")
    print("  Investigar regressões antes de atualizar threshold oficial.")
    BASELINE_UPDATED = False
elif RUNTIME_LABEL != "a100":
    print(f"⚠ UPDATE_BASELINE=True mas hardware é {RUNTIME_LABEL.upper()}, não A100.")
    print("  Baseline oficial jax_gpu_a100 só deve ser atualizada em A100.")
    print("  Para registrar T4 separadamente, edite jax_gpu_t4 manualmente.")
    BASELINE_UPDATED = False
else:
    # ── Backup da baseline anterior antes de sobrescrever ─────────────────────
    # Tenta Drive primeiro; fallback para /tmp se Drive indisponível.
    # Backup TEM que existir antes do overwrite — atomicidade.
    _backup_ts = datetime.datetime.utcnow().strftime("%Y%m%d_%H%M%S")
    _backup_path_primary = Path(OUTPUT_DRIVE_DIR) / f"perf_baseline_backup_{_backup_ts}.json"
    _backup_path_fallback = Path("/tmp") / f"perf_baseline_backup_{_backup_ts}.json"
    _backup_path = None
    _baseline_text_before = BASELINE_PATH.read_text(encoding="utf-8")
    for _candidate in (_backup_path_primary, _backup_path_fallback):
        try:
            _candidate.parent.mkdir(parents=True, exist_ok=True)
            _candidate.write_text(_baseline_text_before, encoding="utf-8")
            _backup_path = _candidate
            print(f"✓ Backup salvo: {_backup_path}")
            break
        except (OSError, PermissionError) as _bk_exc:
            print(f"  [WARN] Backup falhou em {_candidate}: {_bk_exc}")
            continue
    if _backup_path is None:
        BASELINE_UPDATED = False
        raise RuntimeError(
            "Não foi possível criar backup da baseline em Drive nem em /tmp. "
            "Abortando atualização para preservar atomicidade — baseline original intacta."
        )

    # ── Ler versão dinamicamente do __version__ (ADR-0001 R1: sem hardcode) ───
    # Lê do código clonado em vez de cravar string — versões são atribuídas no
    # primeiro commit da sprint (ver docs/ROADMAP.md §0).
    try:
        from importlib.metadata import version as _pkg_version
        _new_version_label = _pkg_version("geosteering_ai")
    except Exception:
        _new_version_label = f"unknown ({GIT_COMMIT[:7]})"

    # ── Atualizar cenários A, B, E in-place ───────────────────────────────────
    # Preserva campos que não medimos (ratio_vs_numba_a100_local, etc.) mas
    # recalcula throughput, threshold_90pct, latency_ms_per_model.
    for scen_name in ("A", "B", "E"):
        scen_key = f"{scen_name}_hot"
        if scen_name not in t16_results or not t16_results[scen_name]["passed"]:
            continue  # só atualiza cenários que passaram (defensa contra ruído)
        r = t16_results[scen_name]
        new_throughput = r["measured_throughput_mod_h"]
        new_threshold = int(round(new_throughput * 0.90))
        _baseline_a100[scen_key]["throughput_mod_h"] = int(round(new_throughput))
        _baseline_a100[scen_key]["threshold_90pct"]  = new_threshold
        _baseline_a100[scen_key]["latency_ms_per_model"] = round(
            r["measured_median_elapsed_s"] * 1000.0 / r["n_models"], 3
        )
        # Anotar mudança no campo notes (audit trail)
        _existing_notes = _baseline_a100[scen_key].get("notes", "")
        _baseline_a100[scen_key]["notes"] = (
            f"[atualizado pós-O1 @ {GIT_COMMIT[:12]}] " + _existing_notes
        ).strip()

    # ── Atualizar _meta (sem hardcode de versão — ADR-0001 R1) ────────────────
    _baseline_a100["_meta"]["version"] = _new_version_label
    _baseline_a100["_meta"]["commit"] = GIT_COMMIT
    _baseline_a100["_meta"]["measured_at"] = (
        datetime.datetime.utcnow().replace(microsecond=0).isoformat() + "Z"
    )
    _baseline_a100["_meta"]["sprint"] = "Sprint O0/O1 Quick Wins (ver docs/ROADMAP.md §0)"
    _baseline_a100["_meta"]["notes"] = (
        f"Baseline atualizada pós Sprint O1 Quick Wins. "
        f"Threshold 90% recalculado em A/B/E. Cenários C-H preservados da medição anterior. "
        + _baseline_a100["_meta"].get("notes", "")
    ).strip()

    # ── Escrever baseline atualizada ──────────────────────────────────────────
    _new_text = json.dumps(_baseline_full, indent=2, ensure_ascii=False) + "\n"
    BASELINE_PATH.write_text(_new_text, encoding="utf-8")

    # Cópia para o Drive (audit trail permanente) — fallback /tmp se Drive falhar
    _drive_new_path_primary = Path(OUTPUT_DRIVE_DIR) / f"perf_baseline_o1_post_update_{_backup_ts}.json"
    _drive_new_path_fallback = Path("/tmp") / f"perf_baseline_o1_post_update_{_backup_ts}.json"
    _drive_new_path = None
    for _candidate in (_drive_new_path_primary, _drive_new_path_fallback):
        try:
            _candidate.write_text(_new_text, encoding="utf-8")
            _drive_new_path = _candidate
            break
        except OSError as _exc:
            print(f"  [WARN] Cópia falhou em {_candidate}: {_exc}")

    BASELINE_UPDATED = True
    print(f"✓ Baseline atualizada: {BASELINE_PATH}")
    if _drive_new_path:
        print(f"✓ Cópia auditável em:  {_drive_new_path}")
    print(f"  Versão registrada: {_new_version_label}")
    print(f"  Novos thresholds 90%:")
    for scen_name in ("A", "B", "E"):
        if scen_name in t16_results and t16_results[scen_name]["passed"]:
            print(f"    {scen_name}: {_baseline_a100[f'{scen_name}_hot']['threshold_90pct']:>14,} mod/h")


## § 8 — Exportar resultados consolidados para Drive

JSON único com:
- Configuração do run (commit, branch, hardware, env vars)
- Pytest summary completo (passed/failed/skipped + durations)
- Quick Wins #1–#9 (label, ok, detail)
- T1.6 — throughput medido vs baseline por cenário
- Status BASELINE_UPDATED


In [ ]:
ts = datetime.datetime.utcnow().strftime("%Y%m%d_%H%M%S")
os.makedirs(OUTPUT_DRIVE_DIR, exist_ok=True)
out_fname = f"sprint_o0_o1_validation_{RUNTIME_LABEL}_{ts}.json"
out_path  = Path(OUTPUT_DRIVE_DIR) / out_fname

output = {
    "sprint":          "O0+O1 (Quick Wins JAX GPU)",
    "template":        "validate_sprint_o0_o1_gpu.ipynb",
    "git_branch":      GIT_BRANCH,
    "git_commit":      GIT_COMMIT,
    "runtime":         RUNTIME_LABEL,
    "timestamp_utc":   ts,
    "jax_version":     jax.__version__,
    "config": {
        "n_runs_hot":      N_RUNS_HOT,
        "seed":            SEED,
        "update_baseline": UPDATE_BASELINE,
        "t16_gate_scenarios": T16_GATE_SCENARIOS,
    },
    "env_vars": {
        "JAX_ENABLE_X64":               os.environ.get("JAX_ENABLE_X64"),
        "JAX_PLATFORMS":                os.environ.get("JAX_PLATFORMS"),
        "JAX_COMPILATION_CACHE_DIR":    os.environ.get("JAX_COMPILATION_CACHE_DIR"),
        "XLA_FLAGS":                    os.environ.get("XLA_FLAGS"),
        "XLA_PYTHON_CLIENT_PREALLOCATE": os.environ.get("XLA_PYTHON_CLIENT_PREALLOCATE"),
        "XLA_PYTHON_CLIENT_MEM_FRACTION": os.environ.get("XLA_PYTHON_CLIENT_MEM_FRACTION"),
    },
    "gates": {
        "gpu_suite":     {"pass": GPU_SUITE_PASS, "summary": _pytest_report["summary"]},
        "quick_wins":    {"pass": QW_ALL_PASS,
                          "n_pass": _qw_n_pass,
                          "n_total": len(qw_checks)},
        "t16":           {"pass": T16_GATE_PASS,
                          "n_pass": sum(r["passed"] for r in t16_results.values()),
                          "n_total": len(t16_results)},
    },
    "pytest_duration_sec": _pytest_report.get("duration", 0.0),
    "quick_wins_details": [
        {"label": label, "ok": ok, "detail": str(detail)}
        for label, ok, detail in qw_checks
    ],
    "t16_results":       t16_results,
    "baseline_updated":  BASELINE_UPDATED,
}

with out_path.open("w") as _f:
    json.dump(output, _f, indent=2, ensure_ascii=False, default=str)

print(f"✓ Resultados exportados: {out_path}")
print(f"  Tamanho: {out_path.stat().st_size / 1024:.1f} KB")


## § 9 — Decisão Go / No-Go para merge em `main`

Consolida os três gates e emite a decisão final. **Critério de aprovação:**
*todos* os três gates PASS — GPU suite (0 FAILED + ≥1 PASS) + Quick Wins (8/8)
+ T1.6 (3/3 cenários ≥ threshold).


In [ ]:
print("=" * 80)
print(f"SPRINT O0/O1 — VALIDAÇÃO JAX GPU CONCLUÍDA")
print(f"Branch  : {GIT_BRANCH}")
print(f"Commit  : {GIT_COMMIT[:12]}")
print(f"Hardware: {RUNTIME_LABEL.upper()} (JAX {jax.__version__})")
print(f"Data UTC: {ts}")
print("=" * 80)

_gates = [
    ("Gate 1 — Suite GPU (0 FAILED + ≥1 PASS)", GPU_SUITE_PASS,
     f"{_pytest_report['summary'].get('passed', 0)} PASS / "
     f"{_pytest_report['summary'].get('failed', 0)} FAIL / "
     f"{_pytest_report['summary'].get('skipped', 0)} SKIP"),
    ("Gate 2 — Quick Wins #1–#9", QW_ALL_PASS,
     f"{_qw_n_pass}/{len(qw_checks)} confirmados"),
    ("Gate 3 — T1.6 throughput pós-O1 vs baseline A100", T16_GATE_PASS,
     f"{sum(r['passed'] for r in t16_results.values())}/{len(t16_results)} cenários ≥ threshold 90%"),
]

for label, ok, detail in _gates:
    status = "✓ PASS" if ok else "✗ FAIL"
    print(f"  {status} {label:55s}  ({detail})")

ALL_GATES_PASS = all(ok for _, ok, _ in _gates)
print()
print("-" * 80)
if ALL_GATES_PASS:
    print("DECISÃO: ✓ GO — Sprint O0/O1 APROVADO para merge em main")
    print()
    print("Próximos passos (versões/tags conforme docs/ROADMAP.md §0 — ADR-0001 R1/R2):")
    print("  1. git checkout main && git merge --no-ff feat/sprint-o1-quick-wins")
    print("  2. Atribuir tag vX.Y conforme ROADMAP §0 (não cravar string aqui — ADR-0001)")
    print("  3. git push origin main --tags")
    print(f"  4. Atualizar docs/CHANGELOG.md com link para {out_fname}")
    print("  5. Iniciar próxima sprint conforme docs/ROADMAP.md §0 backlog priorizado")
else:
    print("DECISÃO: ✗ NO-GO — corrigir gates falhos antes do merge")
    print()
    for label, ok, detail in _gates:
        if not ok:
            print(f"  ► {label}")
            print(f"      {detail}")
print("-" * 80)
print(f"\nResultados salvos em: {out_path}")


## § A — Profiling Roofline (opt-in, A100 recomendado)

**Objetivo**: identificar se o bottleneck do simulador JAX GPU é **compute-bound**,
**memory-bound** ou **latency-bound**, para guiar Sprint O2+.

**Como usar:**
1. Execute esta célula em A100 (T4 também funciona com menos resolução de timing)
2. Download da pasta `/tmp/jax_trace_o1/` (contém `.json.gz` do trace XLA)
3. Abra `.json.gz` em https://ui.perfetto.dev/
4. Inspecione: compute vs HBM vs kernel launches

Não afeta os gates acima — é diagnóstico para Sprint O2.


In [ ]:
# Profile roofline JAX GPU pós-O1 em cenário E (worst case dominante n_pos=600)
from jax import profiler

TRACE_DIR_O1 = Path("/tmp/jax_trace_o1")
TRACE_DIR_O1.mkdir(parents=True, exist_ok=True)
print(f"Trace dir: {TRACE_DIR_O1}")

# Modelo sintético reproduzível — match T1.6 cenário E
_rng = np.random.default_rng(42)
_n_models_prof = 50
_n_pos_prof = 600
_rho_h_prof = _rng.uniform(1.0, 100.0, size=(_n_models_prof, 3)).astype(np.float64)
_rho_v_prof = _rho_h_prof.copy()
_esp_prof = np.full((_n_models_prof, 1), 5.0, dtype=np.float64)
_pos_prof = np.linspace(-5.0, 5.0, _n_pos_prof).astype(np.float64)
_kwargs_prof = dict(frequencies_hz=[20000.0], tr_spacings_m=[1.0], dip_degs=[0.0])

# Warmup (descarta cold-start XLA compile do trace)
print("Warmup (descarta cold-start)...")
_ = simulate_multi_jax_batched(
    _rho_h_prof, _rho_v_prof, _esp_prof, _pos_prof, **_kwargs_prof
).H_tensor.shape

# Trace 1 batch hot
print("Tracing batch hot (cenário E, pós-O1)...")
with profiler.trace(str(TRACE_DIR_O1), create_perfetto_link=False):
    _t0 = time.perf_counter()
    _res_prof = simulate_multi_jax_batched(
        _rho_h_prof, _rho_v_prof, _esp_prof, _pos_prof, **_kwargs_prof
    )
    _ = _res_prof.H_tensor.shape
    _elapsed_prof = time.perf_counter() - _t0

_th_prof = _n_models_prof / _elapsed_prof * 3600.0
print(f"\n✓ Trace concluído:")
print(f"  Tempo: {_elapsed_prof*1000:.2f} ms")
print(f"  Throughput: {_th_prof:,.0f} mod/h")
_trace_files = list(TRACE_DIR_O1.rglob('*.json.gz'))
print(f"  Arquivos: {len(_trace_files)} (.json.gz)")
print(f"\nPróximos passos:")
print(f"  1. Download {TRACE_DIR_O1}")
print(f"  2. Abrir .json.gz em https://ui.perfetto.dev/")
print(f"  3. Documentar findings em docs/reports/v2.43_jax_profile_o1_a100.md")
